# Production model — ETF peer-relative direction signal

Self-contained walkthrough of the deployed model. Predicts whether each US ETF
will **outperform / underperform its category peers** over the next **10 trading
days**, peer-relative (leave-one-out category mean → per-day terciles).

**Architecture** — a blend of two models, each producing a per-fund score, combined
after daily z-scoring:

1. **Two-stage XGBoost** (`stage1.ubj`, `stage2.ubj`)
   - stage 1 = *extreme gate*: P(fund lands in either 15% tail vs the neutral 70%)
   - stage 2 = *direction*: P(top tail | extreme), tuned (depth 6)
   - `s_xgb = P_extreme * (2*P_dir - 1)`
2. **CatGNN** (`catgnn.pt`) — peer-graph net: 2 rounds of attention message-passing
   over same-category cliques, residual to the raw features. Lets each fund's score
   depend on what its peers look like *that same day* — information the trees can't use.

`score = w*z(s_xgb) + (1-w)*z(s_gnn)`, w = 0.2 (chosen on validation rank IC only).

Model files live in `../model_bundle/` (see `MODEL_CARD.md`). This notebook LOADS them
and runs inference — to retrain from raw data run `experiments/train_production.py`.

In [1]:
import sys, pathlib, json
import numpy as np, pandas as pd
import torch, torch.nn as nn
import xgboost as xgb

root = pathlib.Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path[:0] = [str(root), str(root / 'experiments')]
from exp_harness import load_sweep_dataset, load_universe   # data loading only

BUNDLE = root / 'model_bundle'
cfg = json.loads((BUNDLE / 'config.json').read_text())
features, CATS, W = cfg['features'], cfg['categories'], cfg['w_xgb']
HORIZON, HID = cfg['horizon'], cfg['gnn_hidden']
print(f"features={len(features)}  categories={len(CATS)}  blend w_xgb={W}  horizon={HORIZON}d")
print('trained:', cfg['trained'])

features=81  categories=36  blend w_xgb=0.2  horizon=10d
trained: 2026-07-20T11:33:49


## 1. Load the data (features + label, 2026 held out)

`load_sweep_dataset` assembles the 81-feature design matrix from `experiments/features/*.parquet`
(built from `data/raw/market_data.parquet`), attaches the 10-day peer-relative label, and
returns the embargoed 1-year-rolling-window masks. We only need the **test** rows here.

In [2]:
d, feat, fit_m, val_m, te_m, _ = load_sweep_dataset(horizon=HORIZON, window_years=1)
assert feat == features, 'feature set drift vs saved config!'
dates_all = d.index.get_level_values('date')
print('test rows:', int(te_m.sum()), '| days:', dates_all[te_m].nunique())

test rows: 178560 | days: 120


## 2. Two-stage XGBoost score

Plain XGBoost boosters — no custom code to load. `s_xgb = P_extreme * (2*P_dir - 1)`.

In [3]:
b1, b2 = xgb.Booster(), xgb.Booster()
b1.load_model(str(BUNDLE / 'stage1.ubj'))
b2.load_model(str(BUNDLE / 'stage2.ubj'))
dm = xgb.DMatrix(d.loc[te_m, features], feature_names=features)
s_xgb = b1.predict(dm) * (2.0 * b2.predict(dm) - 1.0)
print('s_xgb range:', round(float(s_xgb.min()), 3), '..', round(float(s_xgb.max()), 3))

s_xgb range: -0.823 .. 0.686


## 3. CatGNN peer-graph score

Architecture inlined below (identical to `experiments/nn_common.py`, which the saved
weights were trained with). `attnpool_cat` does the message passing: an
attention-weighted mean of node embeddings **within each category**, computed with
scatter ops (a same-category clique graph, without materializing edges).

In [4]:
def attnpool_cat(h, cid, attn_w):
    """Attention-weighted mean of h within each category (message passing)."""
    e = (h * attn_w).sum(-1)
    emax = torch.full((int(cid.max()) + 1,), -torch.inf, device=h.device)
    emax = emax.scatter_reduce(0, cid, e, reduce='amax')
    a = torch.exp(e - emax[cid])
    denom = torch.zeros_like(emax).scatter_add(0, cid, a)
    a = (a / denom[cid]).unsqueeze(-1)
    pooled = torch.zeros(int(cid.max()) + 1, h.shape[1], device=h.device)
    pooled = pooled.index_add(0, cid, h * a)
    return pooled[cid]


class CatGNN(nn.Module):
    """2 rounds of attention message passing over the category graph,
    residual to the raw features, scalar bullishness score per fund."""
    def __init__(self, nf, hid=HID):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(nf, hid), nn.GELU())
        self.attn1 = nn.Parameter(torch.randn(hid) / hid ** 0.5)
        self.mp1 = nn.Sequential(nn.Linear(2 * hid, hid), nn.GELU(), nn.Dropout(0.2))
        self.n1 = nn.LayerNorm(hid)
        self.attn2 = nn.Parameter(torch.randn(hid) / hid ** 0.5)
        self.mp2 = nn.Sequential(nn.Linear(2 * hid, hid), nn.GELU(), nn.Dropout(0.2))
        self.n2 = nn.LayerNorm(hid)
        self.head = nn.Sequential(nn.Linear(hid + nf, hid), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(hid, 1))

    def forward(self, x, cid):
        z = self.inp(x)
        z = self.n1(z + self.mp1(torch.cat([z, attnpool_cat(z, cid, self.attn1)], -1)))
        z = self.n2(z + self.mp2(torch.cat([z, attnpool_cat(z, cid, self.attn2)], -1)))
        return self.head(torch.cat([z, x], -1)).squeeze(-1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gnn = CatGNN(len(features), HID).to(device)
gnn.load_state_dict(torch.load(BUNDLE / 'catgnn.pt', map_location=device, weights_only=True))
gnn.eval()
print('CatGNN loaded on', device)

CatGNN loaded on cuda


In [5]:
# standardize features with the saved scaler (fit-rows stats), map categories to ids
scaler = pd.read_parquet(BUNDLE / 'scaler.parquet')
Xn = ((d[features].fillna(scaler['median']) - scaler['mean']) / scaler['std']) \
        .clip(-5, 5).fillna(0.0).to_numpy(dtype='float32')

_, cat = load_universe()
cat_map = {c: i for i, c in enumerate(CATS)}
cat_codes = np.array([cat_map.get(c, len(CATS)) for c in
                      d.index.get_level_values('ticker').map(cat)])

# score day-by-day (one graph per trading day)
te_idx = np.flatnonzero(te_m)
s_gnn = np.empty(len(te_idx), dtype='float32')
grp = pd.DataFrame({'i': te_idx}, index=dates_all[te_idx]).groupby(level=0)
pos = 0
with torch.no_grad():
    for _, g in grp:
        ix = g['i'].to_numpy()
        x = torch.from_numpy(Xn[ix]).to(device)
        cid = torch.from_numpy(cat_codes[ix].astype(np.int64)).to(device)
        s_gnn[pos:pos + len(ix)] = gnn(x, cid).cpu().numpy()
        pos += len(ix)
print('s_gnn range:', round(float(s_gnn.min()), 3), '..', round(float(s_gnn.max()), 3))

s_gnn range: -0.49 .. 0.497


## 4. Blend + 5. evaluate & emit the signal

Blend the two scores after **per-day** z-scoring, then rank funds each day.

In [6]:
te = pd.DataFrame({'s_xgb': s_xgb, 's_gnn': s_gnn,
                   'rel': d.loc[te_m, 'rel'].to_numpy(),
                   'target': d.loc[te_m, 'target'].to_numpy()},
                  index=d.index[te_m])
g = te.groupby(level='date')
def zc(col):
    return (te[col] - g[col].transform('mean')) / g[col].transform('std').replace(0, 1)
te['score'] = W * zc('s_xgb') + (1 - W) * zc('s_gnn')

# daily rank IC of score vs realized peer-relative excess
ic = te.groupby(level='date').apply(
    lambda x: x['score'].corr(x['rel'], method='spearman')).mean()
print(f'2026 daily rank IC: {ic:.4f}')

# top/bottom 3% conviction calls per day
te['med'] = g['score'].transform('median')
te['conv'] = (te['score'] - te['med']).abs()
picks = te.groupby(level='date', group_keys=False).apply(
    lambda x: x.nlargest(max(int(len(x) * 0.03), 1), 'conv'))
picks['call'] = np.where(picks['score'] >= picks['med'], 'OVER', 'UNDER')
hit = (((picks['call'] == 'OVER') & (picks['rel'] > 0)) |
       ((picks['call'] == 'UNDER') & (picks['rel'] < 0))).mean()
print(f"gated calls: {len(picks)}  |  directional hit rate: {hit:.3f}")

2026 daily rank IC: 0.0720


gated calls: 5280  |  directional hit rate: 0.567

In [7]:
# latest-day ranked signal (what predict_signal.py emits)
last = dates_all[te_m].max()
day = te[te.index.get_level_values('date') == last].copy()
day['ticker'] = day.index.get_level_values('ticker')
print(f'signal for {last.date()}  (next {HORIZON} trading days, peer-relative)\n')
print('--- top 10 predicted OUTPERFORM ---')
print(day.nlargest(10, 'score')[['ticker', 'score']].to_string(index=False))
print('\n--- top 10 predicted UNDERPERFORM ---')
print(day.nsmallest(10, 'score')[['ticker', 'score']].to_string(index=False))

signal for 2026-06-25  (next 10 trading days, peer-relative)

--- top 10 predicted OUTPERFORM ---
ticker    score
   BTC 2.941619
  SFYF 2.614262
  FDNI 2.542602
   REK 2.535721
   RLY 2.503233
  ROKT 2.501658
  KARS 2.493929
   DRV 2.461787
  HAIL 2.429410
   AGQ 2.395876

--- top 10 predicted UNDERPERFORM ---
ticker     score
  GDXD -3.068797
  DUST -2.930691
   DRN -2.855309
  USDU -2.833395
   URE -2.830973
  YANG -2.788530
   FXP -2.702752
    SH -2.633683
  EJUL -2.627028
   GLL -2.603648


## 6. Retraining reference

The training loop that produced these artifacts lives in `experiments/train_production.py`
(two-stage XGB + `experiments/nn_common.train_catgnn` for the GNN + validation blend-weight
search). To regenerate the bundle from raw data:

```bash
python experiments/train_production.py     # writes experiments/production/*
python experiments/predict_signal.py       # today's ranked calls
```

Backtest vs SPY / equal-weight: `notebooks/05_ensemble_vs_spy.ipynb`.